# Assignment 2: MLP for Iris classification
Design and implement a Multilayer Perceptron, evaluate with accuracy and confusion matrix.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

tf.random.set_seed(42); np.random.seed(42)

## 1. Data

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
class_names = iris.target_names
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
print(X_train.shape, X_test.shape)

## 2. Model design
4 inputs -> Dense(16, ReLU) -> Dense(8, ReLU) -> Dense(3, softmax). Loss: sparse categorical cross-entropy, optimizer: Adam.

In [ ]:
model = keras.Sequential([
    keras.layers.Input((4,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(3, activation="softmax"),
])
model.compile(optimizer=keras.optimizers.Adam(0.01), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 3. Training

In [ ]:
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)
hist = model.fit(X_train, y_train, validation_split=0.2, epochs=200, batch_size=16, callbacks=[es], verbose=0)
print("Stopped at epoch:", len(hist.history["loss"]))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="val"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(hist.history["accuracy"], label="train"); ax[1].plot(hist.history["val_accuracy"], label="val"); ax[1].set_title("Accuracy"); ax[1].legend()
plt.show()

## 4. Evaluation

In [ ]:
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, target_names=class_names))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues")
plt.title("Confusion matrix (test set)"); plt.show()

## 5. (Optional) Effect of hidden-layer size

In [ ]:
rows = []
for units in [4, 8, 16, 32, 64]:
    m = keras.Sequential([keras.layers.Input((4,)), keras.layers.Dense(units, activation="relu"), keras.layers.Dense(units, activation="relu"), keras.layers.Dense(3, activation="softmax")])
    m.compile(optimizer=keras.optimizers.Adam(0.01), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(X_train, y_train, epochs=100, batch_size=16, verbose=0)
    rows.append((units, m.evaluate(X_test, y_test, verbose=0)[1]))
pd.DataFrame(rows, columns=["hidden units", "test accuracy"])

## Conclusion
The MLP reaches high accuracy on Iris (typically 93-100%); the confusion matrix shows that setosa is perfectly separable and any errors are between versicolor and virginica.